# YOLO Polygon → CVAT XML Converter
Converts a folder of YOLO-format `.txt` annotation files into a single CVAT `annotations.xml`.

**YOLO format:** `class_id x1 y1 x2 y2 ... xn yn` (normalized coords, one object per line)  
**CVAT format:** XML with `<polygon points="x1,y1;x2,y2;...">` in absolute pixel coords

In [5]:
# ── Configuration ──────────────────────────────────────────────────────────────

LABELS_DIR   = "/home/infinitydreamer/Desktop/SceneStructNet/dataset/labels/background_seg/test"        # folder containing YOLO .txt files
IMAGES_DIR   = "/home/infinitydreamer/Desktop/SceneStructNet/dataset/images/test"        # folder containing corresponding images (to read W×H)
OUTPUT_XML   = "./annotations.xml"

# Map class index → label name  (extend as needed)
CLASS_NAMES  = {
    0: "background",
}

# Image extensions to search for (order = priority)
IMAGE_EXTS   = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# CVAT meta fields (optional, edit freely)
JOB_ID       = "1"
SUBSET       = "default"
# ───────────────────────────────────────────────────────────────────────────────

In [6]:
from pathlib import Path
from xml.etree import ElementTree as ET
from xml.dom import minidom
from datetime import datetime, timezone

try:
    from PIL import Image as PILImage
    USE_PIL = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "Pillow", "-q"])
    from PIL import Image as PILImage
    USE_PIL = True

print("Dependencies OK")

Dependencies OK


In [7]:
def find_image(stem: str, images_dir: str) -> Path | None:
    for ext in IMAGE_EXTS:
        for candidate in Path(images_dir).glob(f"{stem}{ext}"):
            return candidate
        for candidate in Path(images_dir).glob(f"{stem}{ext.upper()}"):
            return candidate
    return None


def get_image_size(image_path: Path) -> tuple[int, int]:
    with PILImage.open(image_path) as img:
        return img.width, img.height


def yolo_to_cvat_points(coords: list[float], width: int, height: int) -> str:
    """Convert flat [x1,y1,x2,y2,...] normalised list → 'x1,y1;x2,y2;...' absolute."""
    pairs = []
    for i in range(0, len(coords), 2):
        px = round(coords[i] * width, 2)
        py = round(coords[i + 1] * height, 2)
        pairs.append(f"{px},{py}")
    return ";".join(pairs)


def parse_yolo_file(txt_path: Path) -> list[dict]:
    annotations = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = list(map(float, line.split()))
            class_id = int(parts[0])
            coords   = parts[1:]
            if len(coords) < 6 or len(coords) % 2 != 0:
                print(f"  ⚠ Skipping malformed line in {txt_path.name}: {line[:60]}")
                continue
            annotations.append({"class_id": class_id, "coords": coords})
    return annotations


print("Helper functions defined")

Helper functions defined


In [8]:
txt_files = sorted(Path(LABELS_DIR).glob("*.txt"))
print(f"Found {len(txt_files)} YOLO label file(s) in '{LABELS_DIR}'")
if not txt_files:
    raise FileNotFoundError(f"No .txt files found in {LABELS_DIR}")

Found 31 YOLO label file(s) in '/home/infinitydreamer/Desktop/SceneStructNet/dataset/labels/background_seg/test'


In [9]:
# ── Build CVAT XML tree ────────────────────────────────────────────────────────

now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f+00:00")

root = ET.Element("annotations")
ET.SubElement(root, "version").text = "1.1"

meta = ET.SubElement(root, "meta")
job  = ET.SubElement(meta, "job")
ET.SubElement(job, "id").text           = JOB_ID
ET.SubElement(job, "size").text         = str(len(txt_files))
ET.SubElement(job, "mode").text         = "annotation"
ET.SubElement(job, "overlap").text      = "0"
ET.SubElement(job, "bugtracker").text   = ""
ET.SubElement(job, "created").text      = now_str
ET.SubElement(job, "updated").text      = now_str
ET.SubElement(job, "subset").text       = SUBSET
ET.SubElement(job, "start_frame").text  = "0"
ET.SubElement(job, "stop_frame").text   = str(len(txt_files) - 1)
ET.SubElement(job, "frame_filter").text = ""

labels_el = ET.SubElement(job, "labels")
seen_classes: set[int] = set()

# Pre-scan all txt files to collect used class ids
for txt_path in txt_files:
    for ann in parse_yolo_file(txt_path):
        seen_classes.add(ann["class_id"])

for cid in sorted(seen_classes):
    label_el = ET.SubElement(labels_el, "label")
    ET.SubElement(label_el, "n").text          = CLASS_NAMES.get(cid, f"class_{cid}")
    ET.SubElement(label_el, "color").text      = "#ffff00"
    ET.SubElement(label_el, "type").text       = "polygon"
    ET.SubElement(label_el, "attributes").text = ""

ET.SubElement(meta, "dumped").text = now_str

# ── Add image entries ──────────────────────────────────────────────────────────

skipped = []
converted = 0

for frame_id, txt_path in enumerate(txt_files):
    stem       = txt_path.stem
    image_path = find_image(stem, IMAGES_DIR)

    if image_path is None:
        print(f"  ⚠  No image found for '{stem}' — skipping")
        skipped.append(stem)
        continue

    width, height = get_image_size(image_path)
    annotations   = parse_yolo_file(txt_path)

    img_el = ET.SubElement(root, "image")
    img_el.set("id",     str(frame_id))
    img_el.set("name",   image_path.name)
    img_el.set("width",  str(width))
    img_el.set("height", str(height))

    for ann in annotations:
        label_name = CLASS_NAMES.get(ann["class_id"], f"class_{ann['class_id']}")
        points_str = yolo_to_cvat_points(ann["coords"], width, height)

        poly_el = ET.SubElement(img_el, "polygon")
        poly_el.set("label",    label_name)
        poly_el.set("source",   "manual")
        poly_el.set("occluded", "0")
        poly_el.set("points",   points_str)
        poly_el.set("z_order",  "0")

    converted += 1

print(f"\n✓ Converted : {converted} image(s)")
if skipped:
    print(f"✗ Skipped   : {len(skipped)} — {skipped}")


✓ Converted : 31 image(s)


In [10]:
# ── Write pretty-printed XML ───────────────────────────────────────────────────

raw_xml   = ET.tostring(root, encoding="unicode")
pretty    = minidom.parseString(raw_xml).toprettyxml(indent="  ")
# Remove the extra <?xml?> declaration added by toprettyxml
lines     = pretty.split("\n")
final_xml = '<?xml version="1.0" encoding="utf-8"?>\n' + "\n".join(lines[1:])

Path(OUTPUT_XML).write_text(final_xml, encoding="utf-8")
print(f"Saved → {Path(OUTPUT_XML).resolve()}")

Saved → /home/infinitydreamer/Desktop/SceneStructNet/notebooks/annotations.xml


## Quick sanity check — print first image entry

In [11]:
tree = ET.parse(OUTPUT_XML)
images = tree.findall("image")
print(f"Total <image> entries: {len(images)}")
if images:
    first = images[0]
    print(f"\nFirst image: id={first.get('id')} name={first.get('name')} "
          f"{first.get('width')}×{first.get('height')}")
    for p in first.findall("polygon"):
        pts = p.get('points', '')
        preview = pts[:80] + ("..." if len(pts) > 80 else "")
        print(f"  polygon label={p.get('label')} points={preview}")

Total <image> entries: 31

First image: id=0 name=gg_architecture_094ffe9a22fb863211de7386157fd9da6558a86e.jpg 4000×2418
  polygon label=background points=3350.0,2405.0;3312.0,2385.0;3212.0,2343.0;3151.0,2312.0;3106.0,2297.0;3092.0,228...
